# **Analisis Sentimental Aplikasi Playstore (Aplikasi myBCA)**

## Installasi Library

In [ ]:
!pip install google-play-scraper Sastrawi gensim gradio scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.4 MB/s eta 0:00:00


## 1. Data Collection

In [ ]:
from google_play_scraper import reviews_all, Sort
import pandas as pd

In [ ]:
data = reviews_all(
    'com.bca.mybca.omni.android',
    sleep_milliseconds=0, # Atur jeda jika terkena limit/block, default 0
    lang='id',
    country='id',
    sort=Sort.NEWEST
)

In [ ]:
df = pd.DataFrame(data)
df = df[['content', 'score']] # Ambil teks dan rating saja

# TAMPILKAN DATA SEBELUM DIHAPUS
print(f"Total data diambil (Original): {len(df)} baris")

Total data diambil (Original): 33686 baris


In [ ]:
# Labeling: Positif (1) untuk rating 4-5, Negatif (0) untuk rating 1-2
df['label'] = df['score'].apply(lambda x: 1 if x >= 4 else (0 if x <= 2 else None))
df = df.dropna(subset=['label']) # Hapus yang rating 3 (Netral)

In [ ]:
# Banyak data setelah menghapus rating 3 (netral)
print(f"Total data diambil setelah penghapusan: {len(df)} baris")
df

Total data diambil setelah penghapusan: 31045 baris


,content,score,label
0,ini saya mau nambahin nomer telp buat bca saya...,4,1.0
1,ok,5,1.0
2,semua BCA milik Allah,5,1.0
3,KENAPA YAAA SUKANYA KEBLOKIR TERUS. ENGGA PERN...,2,0.0
4,"sangat buruk sekali nyesal saya pakek bca, ken...",1,0.0
...,...,...,...
33681,Akhirnya keluar juga,5,1.0
33682,Ayo lengkapin lagi fiturnya,5,1.0
33683,Lebih gampang kelola rekening,5,1.0
33684,"Fiturnya lengkap, 👍👍👍👍",5,1.0


## 2. Text Preprocessing

In [8]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def cleaning_dasar(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+|[^\w\s]|[\d+]', '', text)
    return stopword_remover.remove(text)

print("Step 1: Cleaning & Stopword...")
df['clean_text'] = df['content'].apply(cleaning_dasar)

print("Step 2: Stemming Unik...")
# Ambil kata unik saja agar tidak kerja berkali-kali untuk kata yang sama
unique_words = list(set(' '.join(df['clean_text']).split()))
print(f"Jumlah kata unik yang akan di-stem: {len(unique_words)}")

# Proses stem hanya pada kata unik
word_to_stem = {word: stemmer.stem(word) for word in unique_words}

# Kembalikan ke dataset
def apply_fast_stem(text):
    return ' '.join([word_to_stem.get(w, w) for w in text.split()])

df['clean_text'] = df['clean_text'].apply(apply_fast_stem)
print("Preprocessing Selesai!")

Step 1: Cleaning & Stopword...
Step 2: Stemming Unik...
Jumlah kata unik yang akan di-stem: 19376
Preprocessing Selesai!


In [9]:
display(df[['content', 'clean_text']].head())

,content,clean_text
0,"buruk, sering kesulitan buka mybca... lebih da...",buruk sering sulit buka mybca lebih x dlm hari...
1,good,good
2,ini saya mau nambahin nomer telp buat bca saya...,mau nambahin nomer telp buat bca saya masa iya...
3,ok,
4,semua BCA milik Allah,semua bca milik allah


## 3. Feature Extraction

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

# A. TF-IDF
tfidf_vect = TfidfVectorizer(max_features=2500)
X_tfidf = tfidf_vect.fit_transform(df['clean_text']).toarray()

# B. Word Embedding (Word2Vec)
# Tokenisasi untuk Word2Vec
sentences = [text.split() for text in df['clean_text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def get_vector(text):
    words = text.split()
    vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_w2v = np.array([get_vector(t) for t in df['clean_text']])
y = df['label'].values

## 4. Modeling

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Split data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Inisialisasi Model
nb_model = MultinomialNB()
lr_model = LogisticRegression(max_iter=1000)
svm_model = SVC(kernel='linear')

# Training Model
print("Melatih Model Naive Bayes...")
nb_model.fit(X_train, y_train)

print("Melatih Model Logistic Regression...")
lr_model.fit(X_train, y_train)

print("Melatih Model SVM...")
svm_model.fit(X_train, y_train)

Melatih Model Naive Bayes...
Melatih Model Logistic Regression...
Melatih Model SVM...


SVC(kernel='linear')

## 5. Comparison Model

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def get_metrics(model, X, y_true):
    y_pred = model.predict(X)
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }

# Evaluasi
results = {
    "Naive Bayes": get_metrics(nb_model, X_test, y_test),
    "Logistic Regression": get_metrics(lr_model, X_test, y_test),
    "SVM": get_metrics(svm_model, X_test, y_test)
}

# Tampilkan perbandingan
df_results = pd.DataFrame(results).T
print(df_results)

                     Accuracy  Precision    Recall  F1-Score
Naive Bayes          0.895652   0.939527  0.855692  0.895652
Logistic Regression  0.897907   0.910289  0.892923  0.901522
SVM                  0.900644   0.925089  0.881538  0.902789


In [13]:
df.to_csv('dataset_mybca_cleaned.csv', index=False)
print("Data bersih BERHASIL diamankan!")

Data bersih BERHASIL diamankan!


## 6. Simple Deployment Menggunakan *Gradio*

In [16]:
import gradio as gr
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_remover_gradio = StopWordRemoverFactory().create_stop_word_remover()

def cleaning_khusus_gradio(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+|[^\w\s]|[\d+]', '', text)
    return stopword_remover_gradio.remove(text)

def prediksi_sentimen(teks_review):
    # Bersihkan teks
    teks_bersih = cleaning_khusus_gradio(teks_review)

    # Ubah ke vektor
    teks_vektor = tfidf_vect.transform([teks_bersih]).toarray()

    # Prediksi menggunakan SVM (Hasilnya berupa angka 1.0 atau 0.0)
    hasil_angka = svm_model.predict(teks_vektor)[0]

    # Terjemahkan angka ke bahasa manusia agar tampilannya bagus
    if hasil_angka == 1 or hasil_angka == 1.0:
        return "Sentimen: POSITIF 😃 (Pengguna Puas)"
    else:
        return "Sentimen: NEGATIF 😡 (Pengguna Kecewa/Komplain)"

# 3. Buat Tampilan Website
demo = gr.Interface(
    fn=prediksi_sentimen,
    inputs=gr.Textbox(lines=4, placeholder="Ketik review atau keluhan aplikasi myBCA di sini..."),
    outputs=gr.Textbox(label="Hasil Analisis"),
    title="Aplikasi Analisis Sentimen myBCA",
    description="Coba ketik ulasan, lalu klik 'Submit' untuk melihat hasilnya.",
    theme="default"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c64b2be5448e192486.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c64b2be5448e192486.gradio.live




## 7. Mendownload model

In [17]:
import joblib

# Simpan model
joblib.dump(nb_model, 'naive_bayes_model.pkl')
joblib.dump(lr_model, 'logistic_regression_model.pkl')
joblib.dump(svm_model, 'svm_model.pkl')
joblib.dump(tfidf_vect, 'tfidf_vectorizer.pkl')

print("Model dan TF-IDF Vectorizer berhasil disimpan!")

Model dan TF-IDF Vectorizer berhasil disimpan!
